# RQ3: Polarization and Echo Chambers

Standalone polarization workflow covering complexity, temporal structure, network echo chambers, and ML stance classification.


In [ ]:
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'requirements.txt').exists():
            return path
    return start


ROOT_DIR = find_repo_root()
DATA_DIR = ROOT_DIR / 'data'

def load_labeled_data():
    reddit = pd.read_csv(DATA_DIR / 'reddit_labeled.csv')
    youtube = pd.read_csv(DATA_DIR / 'youtube_labeled.csv')

    for col in ['score', 'post_score', 'post_upvote_ratio', 'user_total_karma', 'controversiality']:
        if col in reddit.columns:
            reddit[col] = pd.to_numeric(reddit[col], errors='coerce')
    for col in ['likeCount', 'replyCount']:
        if col in youtube.columns:
            youtube[col] = pd.to_numeric(youtube[col], errors='coerce')

    reddit_text_col = 'self_text'
    youtube_text_col = 'text'
    if reddit_text_col in reddit.columns:
        reddit['text_length'] = reddit[reddit_text_col].astype(str).str.len()
        reddit['word_count'] = reddit[reddit_text_col].astype(str).str.split().str.len()
    if youtube_text_col in youtube.columns:
        youtube['text_length'] = youtube[youtube_text_col].astype(str).str.len()
        youtube['word_count'] = youtube[youtube_text_col].astype(str).str.split().str.len()
    return reddit, youtube


def add_sentiment_columns(reddit, youtube):
    from textblob import TextBlob
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    vader = SentimentIntensityAnalyzer()

    def vader_scores(text):
        if pd.isna(text) or text == '':
            return {'compound': 0, 'pos': 0, 'neu': 0, 'neg': 0, 'label': 'neutral'}
        scores = vader.polarity_scores(str(text))
        if scores['compound'] >= 0.05:
            label = 'positive'
        elif scores['compound'] <= -0.05:
            label = 'negative'
        else:
            label = 'neutral'
        scores['label'] = label
        return scores

    def textblob_scores(text):
        if pd.isna(text) or text == '':
            return {'polarity': 0, 'subjectivity': 0, 'label': 'neutral'}
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
        if polarity > 0.1:
            label = 'positive'
        elif polarity < -0.1:
            label = 'negative'
        else:
            label = 'neutral'
        return {'polarity': polarity, 'subjectivity': subjectivity, 'label': label}

    reddit_text_col = 'self_text'
    youtube_text_col = 'text'

    for df, text_col in [(reddit, reddit_text_col), (youtube, youtube_text_col)]:
        vader_results = df[text_col].apply(vader_scores)
        df['vader_compound'] = vader_results.apply(lambda x: x['compound'])
        df['vader_positive'] = vader_results.apply(lambda x: x['pos'])
        df['vader_neutral'] = vader_results.apply(lambda x: x['neu'])
        df['vader_negative'] = vader_results.apply(lambda x: x['neg'])
        df['vader_label'] = vader_results.apply(lambda x: x['label'])

        blob_results = df[text_col].apply(textblob_scores)
        df['textblob_polarity'] = blob_results.apply(lambda x: x['polarity'])
        df['textblob_subjectivity'] = blob_results.apply(lambda x: x['subjectivity'])
        df['textblob_label'] = blob_results.apply(lambda x: x['label'])

    return reddit, youtube


def load_sentiment_data():
    reddit, youtube = load_labeled_data()
    return add_sentiment_columns(reddit, youtube)


In [ ]:
from pathlib import Path

# ==================== EDIT THESE PATHS ====================
# Local mode:  INPUT_DIR = Path('../../02_emotional_tone_analysis/outputs')
# Kaggle mode: INPUT_DIR = Path('/kaggle/input/israel-hamas-data')
# =========================================================

INPUT_DIR = Path('../../02_emotional_tone_analysis/outputs')  # <-- EDIT: Module 02 outputs (reddit_with_sentiment.csv, etc.)
OUTPUT_DIR = Path.cwd()                                        # <-- EDIT: where to save outputs

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input directory:  {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## Kaggle Setup & Input Paths

Configure paths and load sentiment data from Module 02.



## Advanced Discourse Analysis


In [ ]:
"""
Phase 4: Advanced Analysis
RQ2: Regression Models & Algorithmic Amplification
RQ3: Narrative Complexity
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
import textstat
import warnings
warnings.filterwarnings('ignore')

# Create output directory

print("="*80)
print("PHASE 4: ADVANCED ANALYSIS")
print("="*80)


In [ ]:
# ============================================================================
# 1. LOAD DATA


In [ ]:
# ============================================================================
print("\nðŸ“Š Loading data...")
reddit_df, youtube_df = load_sentiment_data()

print(f"âœ“ Reddit: {len(reddit_df)} rows")
print(f"âœ“ YouTube: {len(youtube_df)} rows")


In [ ]:
# ============================================================================
# 2. REGRESSION MODELS (RQ2) - REDDIT ONLY


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ2: REGRESSION MODELS (PREDICTING ENGAGEMENT)")
print("="*80)

# Prepare Reddit Data
# We want to predict 'score' based on 'vader_compound' (Sentiment) and 'Label' (Stance)
reg_df = reddit_df[['score', 'vader_compound', 'Label']].dropna()
reg_df['score'] = pd.to_numeric(reg_df['score'], errors='coerce')
reg_df = reg_df.dropna()

print(f"Data points for regression: {len(reg_df)}")

if len(reg_df) > 0:
    # Define model: Score ~ Sentiment + Stance
    # We treat 'Label' as categorical
    model = ols('score ~ vader_compound + C(Label)', data=reg_df).fit()
    
    print("\n--- Regression Results (Score ~ Sentiment + Stance) ---")
    print(model.summary())
    
    # Save results
    
    # Visualization of coefficients
    params = model.params.drop('Intercept')
    conf = model.conf_int().drop('Intercept')
    conf.columns = ['Lower', 'Upper']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    params.plot(kind='barh', xerr=(params - conf['Lower']), ax=ax, color='#3498db')
    ax.set_title('Impact of Stance and Sentiment on Reddit Score', fontsize=14, fontweight='bold')
    ax.set_xlabel('Coefficient Value (Change in Score)')
    ax.axvline(0, color='black', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

else:
    print("âš ï¸ Not enough data for regression analysis.")


In [ ]:
# ============================================================================
# 3. ALGORITHMIC AMPLIFICATION (RQ2) - REDDIT ONLY


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ2: ALGORITHMIC AMPLIFICATION (CONTROVERSIALITY)")
print("="*80)

if 'controversiality' in reddit_df.columns and 'score' in reddit_df.columns:
    # Compare scores for controversial vs non-controversial
    # controversiality is usually 0 or 1
    
    amplification_df = reddit_df[['score', 'controversiality', 'Label']].dropna()
    amplification_df['controversiality'] = amplification_df['controversiality'].astype(int)
    
    # Calculate mean score by controversiality
    mean_scores = amplification_df.groupby('controversiality')['score'].mean()
    print("\nMean Score by Controversiality:")
    print(mean_scores)
    
    # Visualization
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x='controversiality', y='score', data=amplification_df, ax=ax, palette='viridis')
    ax.set_title('Impact of Controversiality on Engagement (Reddit)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Controversial (0=No, 1=Yes)')
    ax.set_ylabel('Average Score')
    ax.set_xticklabels(['Non-Controversial', 'Controversial'])
    plt.tight_layout()
    plt.show()
    
    # Interaction with Stance
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='Label', y='score', hue='controversiality', data=amplification_df, ax=ax, palette='viridis')
    ax.set_title('Controversiality Amplification by Stance', fontsize=14, fontweight='bold')
    ax.set_xlabel('Stance')
    ax.set_ylabel('Average Score')
    plt.legend(title='Controversial', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

else:
    print("âš ï¸ Controversiality or Score data missing.")


In [ ]:
# ============================================================================
# 4. NARRATIVE COMPLEXITY (RQ3)


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ3: NARRATIVE COMPLEXITY (READABILITY & COHERENCE)")
print("="*80)

def calculate_complexity(text):
    """Calculate readability score (Flesch Reading Ease)"""
    if pd.isna(text) or text == '':
        return np.nan
    try:
        return textstat.flesch_reading_ease(str(text))
    except:
        return np.nan

print("Calculating readability scores (this may take a moment)...")
reddit_df['readability'] = reddit_df['self_text'].apply(calculate_complexity)
youtube_df['readability'] = youtube_df['text'].apply(calculate_complexity)

# Filter out invalid scores (textstat can return negative for gibberish)
reddit_df = reddit_df[reddit_df['readability'] > -100]
youtube_df = youtube_df[youtube_df['readability'] > -100]

print(f"Reddit Mean Readability: {reddit_df['readability'].mean():.2f}")
print(f"YouTube Mean Readability: {youtube_df['readability'].mean():.2f}")
print("(Higher score = Easier to read, Lower score = More complex)")

# Visualization: Platform Comparison
fig, ax = plt.subplots(figsize=(10, 6))
data_to_plot = [reddit_df['readability'].dropna(), youtube_df['readability'].dropna()]
ax.boxplot(data_to_plot, labels=['Reddit', 'YouTube'])
ax.set_title('Narrative Complexity: Readability Scores', fontsize=14, fontweight='bold')
ax.set_ylabel('Flesch Reading Ease Score')
plt.tight_layout()
plt.show()

# Visualization: By Stance (Reddit)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='Label', y='readability', data=reddit_df, ax=axes[0], palette='Set2')
axes[0].set_title('Reddit Complexity by Stance', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Readability Score')

# Visualization: By Stance (YouTube)
label_col_yt = 'Label'
sns.boxplot(x=label_col_yt, y='readability', data=youtube_df, ax=axes[1], palette='Set2')
axes[1].set_title('YouTube Complexity by Stance', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Readability Score')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("âœ… ADVANCED ANALYSIS COMPLETE")
print("="*80)


## Structural and Temporal Analysis


In [ ]:
# Temporal activity trends
if 'created_time_dt' in reddit_df.columns:
    reddit_df['year_month'] = reddit_df['created_time_dt'].dt.to_period('M')
    r_temporal = reddit_df.groupby('year_month').size()
    print(f"Reddit monthly volume:\n{r_temporal}")

if 'created_time_dt' in youtube_df.columns:
    youtube_df['year_month'] = youtube_df['created_time_dt'].dt.to_period('M')
    y_temporal = youtube_df.groupby('year_month').size()
    print(f"YouTube monthly volume:\n{y_temporal}")

# Combined temporal plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Monthly Comment Volume (Both Platforms)', fontsize=15, fontweight='bold')

for ax, df, label_col, title, color in [
        (axes[0], reddit_df, 'Label', 'Reddit', '#3498db'),
        (axes[1], youtube_df, 'Label', 'YouTube', '#e74c3c')]:
    if 'year_month' in df.columns:
        total = df.groupby('year_month').size()
        total.index = total.index.astype(str)
        ax.bar(total.index, total.values, color=color, alpha=0.7, edgecolor='black', label='Total')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Month'); ax.set_ylabel('Comment Count')
        ax.legend(title='Series'); ax.grid(alpha=0.3)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


## Temporal Activity Trends

Month-over-month comment volume by platform and stance.



In [ ]:
"""
Phase 4: Structural & Temporal Analysis
RQ2: Network Analysis (User-Post)
RQ3: Response Time & Thread Depth (Volume)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Output directory is already created by advanced_analysis.py, but good to be safe

print("="*80)
print("PHASE 4: STRUCTURAL & TEMPORAL ANALYSIS")
print("="*80)


In [ ]:
# ============================================================================
# 1. LOAD DATA


In [ ]:
# ============================================================================
print("\nðŸ“Š Loading data...")
reddit_df, youtube_df = load_sentiment_data()

print(f"âœ“ Reddit: {len(reddit_df)} rows")
print(f"âœ“ YouTube: {len(youtube_df)} rows")


In [ ]:
# ============================================================================
# 2. NETWORK ANALYSIS (RQ2) - REDDIT


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ2: NETWORK ANALYSIS (USER ACTIVITY)")
print("="*80)

# Since we lack parent_id for full trees, we analyze User-Post activity
if 'author_name' in reddit_df.columns:
    user_counts = reddit_df['author_name'].value_counts()
    top_users = user_counts.head(20)
    
    print("\nTop 10 Most Active Users (Reddit):")
    print(top_users.head(10))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(x=top_users.values, y=top_users.index, ax=ax, palette='viridis')
    ax.set_title('Top 20 Most Active Users (Reddit)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Comments')
    plt.tight_layout()
    plt.show()
    
    # User Stance Consistency (if users comment multiple times)
    # Find users with > 5 comments
    active_users = user_counts[user_counts >= 5].index
    user_stance_df = reddit_df[reddit_df['author_name'].isin(active_users)]
    
    # Calculate dominant stance per user
    user_stance_dominance = user_stance_df.groupby('author_name')['Label'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
    
    print("\nStance Distribution of Active Users (>5 comments):")
    print(user_stance_dominance.value_counts())

else:
    print("âš ï¸ Author name missing for network analysis.")


In [ ]:
# ============================================================================
# 3. RESPONSE TIME ANALYSIS (RQ3) - REDDIT ONLY


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ3: RESPONSE TIME ANALYSIS (IMMEDIACY)")
print("="*80)

if 'created_time' in reddit_df.columns and 'post_created_time' in reddit_df.columns:
    # Convert to datetime
    reddit_df['created_time'] = pd.to_datetime(reddit_df['created_time'], errors='coerce')
    reddit_df['post_created_time'] = pd.to_datetime(reddit_df['post_created_time'], errors='coerce')
    
    # Calculate difference in minutes
    reddit_df['response_time_mins'] = (reddit_df['created_time'] - reddit_df['post_created_time']).dt.total_seconds() / 60
    
    # Filter out negative times (data errors) or extremely long times (archived posts)
    # Let's look at first 48 hours (2880 mins)
    response_df = reddit_df[(reddit_df['response_time_mins'] > 0) & (reddit_df['response_time_mins'] < 2880)]
    
    print(f"Analyzed {len(response_df)} comments within 48 hours of posting.")
    print(f"Median Response Time: {response_df['response_time_mins'].median():.2f} minutes")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(response_df['response_time_mins'], bins=50, kde=True, color='#e74c3c', ax=ax)
    ax.set_title('Distribution of Response Times (Reddit)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time since Post Creation (Minutes)')
    ax.set_xlim(0, 1440) # Show first 24 hours
    plt.tight_layout()
    plt.show()
    
    # Response Time by Stance
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(x='Label', y='response_time_mins', data=response_df, ax=ax, palette='Set2', showfliers=False)
    ax.set_title('Response Time by Stance (First 48h)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Response Time (Minutes)')
    plt.tight_layout()
    plt.show()

else:
    print("âš ï¸ Timestamp data missing for response time analysis.")


In [ ]:
# ============================================================================
# 4. THREAD DEPTH / CONVERSATION VOLUME (RQ3)


In [ ]:
# ============================================================================
print("\n" + "="*80)
print("RQ3: CONVERSATION VOLUME (PROXY FOR DEPTH)")
print("="*80)

# Reddit: Comments per Post
if 'post_id' in reddit_df.columns:
    reddit_vol = reddit_df.groupby('post_id').size()
    print(f"\nReddit: Avg Comments per Post (in dataset): {reddit_vol.mean():.2f}")
else:
    reddit_vol = pd.Series([])

# YouTube: Comments per Video
if 'video id' in youtube_df.columns:
    youtube_vol = youtube_df.groupby('video id').size()
    print(f"YouTube: Avg Comments per Video (in dataset): {youtube_vol.mean():.2f}")
else:
    youtube_vol = pd.Series([])

# Visualization
if len(reddit_vol) > 0 and len(youtube_vol) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Normalize for comparison? Or just raw counts?
    # Raw counts might be misleading if sampling was different.
    # But let's show the distribution.
    
    sns.kdeplot(reddit_vol, label='Reddit (Comments/Post)', shade=True, color='#3498db', ax=ax)
    sns.kdeplot(youtube_vol, label='YouTube (Comments/Video)', shade=True, color='#e74c3c', ax=ax)
    
    ax.set_title('Conversation Volume Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Comments')
    ax.set_xlim(0, 100) # Limit x-axis for readability
    ax.legend()
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("âœ… STRUCTURAL & TEMPORAL ANALYSIS COMPLETE")
print("="*80)


## Network Echo Chambers


In [ ]:
"""
Phase 5: Network Analysis
1. User Interaction Networks (Bipartite User-Post Graph)
2. Echo Chamber Detection (Homophily Index)
3. Information Flow Patterns (Temporal Stance Shifts)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Create output directory

print("="*80)
print("PHASE 5: NETWORK ANALYSIS & ECHO CHAMBERS")
print("="*80)


In [ ]:
# ============================================================================
# 1. LOAD DATA


In [ ]:
# ============================================================================
print("\nðŸ“Š Loading data...")
reddit_df, youtube_df = load_sentiment_data()

# Filter for valid labels
valid_labels = ['P', 'I', 'N']
reddit_df = reddit_df[reddit_df['Label'].isin(valid_labels)]

print(f"âœ“ Reddit Data: {len(reddit_df)} rows")


In [ ]:
# ============================================================================
# 2. USER STANCE PROFILING


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("USER STANCE PROFILING")
print("-"*(60))

if 'author_name' in reddit_df.columns:
    # Count comments per user
    user_counts = reddit_df['author_name'].value_counts()
    active_users = user_counts[user_counts >= 3].index # Users with at least 3 comments
    
    print(f"Total Users: {len(user_counts)}")
    print(f"Active Users (>=3 comments): {len(active_users)}")
    
    # Determine "Dominant Stance" for each active user
    user_profiles = {}
    
    for user in active_users:
        user_comments = reddit_df[reddit_df['author_name'] == user]
        stance_counts = user_comments['Label'].value_counts()
        dominant_stance = stance_counts.idxmax()
        consistency = stance_counts.max() / len(user_comments)
        
        user_profiles[user] = {
            'dominant_stance': dominant_stance,
            'consistency': consistency,
            'total_comments': len(user_comments)
        }
    
    profiles_df = pd.DataFrame.from_dict(user_profiles, orient='index')
    
    # Visualize User Stance Distribution
    plt.figure(figsize=(8, 6))
    sns.countplot(x='dominant_stance', data=profiles_df, palette={'P': '#2ecc71', 'I': '#3498db', 'N': '#95a5a6'})
    plt.title('Dominant Stance of Active Users (Reddit)', fontsize=14, fontweight='bold')
    plt.xlabel('Stance')
    plt.ylabel('Number of Users')
    plt.tight_layout()
    plt.show()
    
    # Visualize Consistency
    plt.figure(figsize=(10, 6))
    sns.histplot(data=profiles_df, x='consistency', hue='dominant_stance', element='step', palette={'P': '#2ecc71', 'I': '#3498db', 'N': '#95a5a6'})
    plt.title('User Stance Consistency (1.0 = Always same stance)', fontsize=14, fontweight='bold')
    plt.xlabel('Consistency Score')
    plt.tight_layout()
    plt.show()
    active_users = []


In [ ]:
# ============================================================================
# 3. ECHO CHAMBER DETECTION (HOMOPHILY)


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("ECHO CHAMBER DETECTION (HOMOPHILY)")
print("-"*(60))

if len(active_users) > 0 and 'post_id' in reddit_df.columns:
    # 1. Determine "Thread Stance" (Majority stance of comments in a thread)
    thread_stance = reddit_df.groupby('post_id')['Label'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'N')
    
    # 2. Calculate Homophily for each user
    # Homophily = % of comments on threads that match the user's dominant stance
    
    homophily_scores = []
    
    for user in active_users:
        user_dom_stance = user_profiles[user]['dominant_stance']
        user_comments = reddit_df[reddit_df['author_name'] == user]
        
        # Get the stance of the threads they commented on
        thread_ids = user_comments['post_id'].unique()
        matching_threads = 0
        total_threads = 0
        
        for tid in thread_ids:
            if tid in thread_stance:
                t_stance = thread_stance[tid]
                total_threads += 1
                if t_stance == user_dom_stance:
                    matching_threads += 1
        
        if total_threads > 0:
            score = matching_threads / total_threads
            homophily_scores.append({
                'user': user,
                'dominant_stance': user_dom_stance,
                'homophily_index': score
            })
            
    homophily_df = pd.DataFrame(homophily_scores)
    
    print(f"Average Homophily Index: {homophily_df['homophily_index'].mean():.4f}")
    
    # Visualize Homophily by Stance
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='dominant_stance', y='homophily_index', data=homophily_df, palette={'P': '#2ecc71', 'I': '#3498db', 'N': '#95a5a6'})
    plt.title('Echo Chamber Effect: Homophily Index by Stance', fontsize=14, fontweight='bold')
    plt.ylabel('Homophily Index (% interaction with same-stance threads)')
    plt.xlabel('User Stance')
    plt.tight_layout()
    plt.show()
    
    # Save high homophily users
    echo_chamber_users = homophily_df[homophily_df['homophily_index'] > 0.8]
    print(f"Users in 'Echo Chambers' (>80% same-stance interaction): {len(echo_chamber_users)} ({len(echo_chamber_users)/len(homophily_df):.1%})")

else:
    print("âš ï¸ Cannot calculate homophily (missing post_id or active users).")


In [ ]:
# ============================================================================
# 4. NETWORK VISUALIZATION (BIPARTITE PROJECTION)


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("NETWORK VISUALIZATION")
print("-"*(60))

if len(active_users) > 0 and 'post_id' in reddit_df.columns:
    # Create a graph of Users connected to Threads
    # We'll filter for top 50 most active users to keep the graph readable
    top_50_users = user_counts.head(50).index
    subset_df = reddit_df[reddit_df['author_name'].isin(top_50_users)]
    
    B = nx.Graph()
    B.add_nodes_from(subset_df['author_name'].unique(), bipartite=0, type='user')
    B.add_nodes_from(subset_df['post_id'].unique(), bipartite=1, type='post')
    
    # Add edges
    edges = list(zip(subset_df['author_name'], subset_df['post_id']))
    B.add_edges_from(edges)
    
    # Project to User-User graph (Users connected if they commented on same post)
    user_nodes = {n for n, d in B.nodes(data=True) if d['type'] == 'user'}
    G = nx.bipartite.weighted_projected_graph(B, user_nodes)
    
    # Draw Graph
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G, k=0.5, iterations=50)
    
    # Color nodes by stance
    node_colors = []
    for node in G.nodes():
        stance = user_profiles[node]['dominant_stance']
        if stance == 'P': node_colors.append('#2ecc71')
        elif stance == 'I': node_colors.append('#3498db')
        else: node_colors.append('#95a5a6')
    
    # Draw
    nx.draw_networkx_nodes(G, pos, node_size=100, node_color=node_colors, alpha=0.8)
    nx.draw_networkx_edges(G, pos, alpha=0.1)
    
    # Create legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Pro-Palestine', markerfacecolor='#2ecc71', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Pro-Israel', markerfacecolor='#3498db', markersize=10),
        Line2D([0], [0], marker='o', color='w', label='Neutral', markerfacecolor='#95a5a6', markersize=10)
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.title('User Interaction Network (Top 50 Users)\nConnected if commented on same post', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("âœ… NETWORK ANALYSIS COMPLETE")
print("="*80)


## ML Stance Classification


In [ ]:
"""
Phase 5: Machine Learning Models
1. Classification models for stance prediction (Ensemble Voting Classifier)
2. Feature importance analysis (Top keywords per stance)
3. Cross-platform prediction (Train on Reddit -> Predict YouTube and vice versa)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Create output directory

print("="*80)
print("PHASE 5: MACHINE LEARNING - STANCE PREDICTION (ADVANCED ENSEMBLE)")
print("="*80)


In [ ]:
# ============================================================================
# 1. LOAD AND PREPARE DATA


In [ ]:
# ============================================================================
print("\nðŸ“Š Loading data...")
reddit_df, youtube_df = load_sentiment_data()

# Standardize columns
reddit_df['text'] = reddit_df['self_text'].fillna('')
youtube_df['text'] = youtube_df['text'].fillna('')

# Filter valid labels (P, I, N)
valid_labels = ['P', 'I', 'N']
reddit_df = reddit_df[reddit_df['Label'].isin(valid_labels)]
youtube_df = youtube_df[youtube_df['Label'].isin(valid_labels)]

# Map labels to full names for better plots
label_map = {'P': 'Pro-Palestine', 'I': 'Pro-Israel', 'N': 'Neutral'}
reddit_df['label_full'] = reddit_df['Label'].map(label_map)
youtube_df['label_full'] = youtube_df['Label'].map(label_map)

print(f"âœ“ Reddit (Filtered): {len(reddit_df)} rows")
print(f"âœ“ YouTube (Filtered): {len(youtube_df)} rows")


In [ ]:
# ============================================================================
# 2. MODEL DEFINITION


In [ ]:
# ============================================================================
def get_ensemble_model():
    """
    Creates a Voting Classifier composed of:
    1. Logistic Regression (Baseline)
    2. Linear SVM (Good for high-dimensional text)
    3. Random Forest (Captures non-linearities)
    4. Gradient Boosting (Iterative correction)
    """
    clf1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    clf2 = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
    clf3 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    clf4 = GradientBoostingClassifier(n_estimators=100, random_state=42)
    
    voting_clf = VotingClassifier(
        estimators=[
            ('lr', clf1), 
            ('svm', clf2), 
            ('rf', clf3), 
            ('gb', clf4)
        ],
        voting='soft'
    )
    return voting_clf


In [ ]:
# ============================================================================
# 3. TRAINING & EVALUATION FUNCTION


In [ ]:
# ============================================================================
def train_evaluate_model(train_df, test_df, train_name, test_name):
    print(f"\nðŸ¤– Training on {train_name}, Testing on {test_name}...")
    
    # Encode labels
    le = LabelEncoder()
    y_train_enc = le.fit_transform(train_df['label_full'])
    y_test_enc = le.transform(test_df['label_full'])
    
    # Vectorize
    tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
    X_train_vec = tfidf.fit_transform(train_df['text'])
    X_test_vec = tfidf.transform(test_df['text'])
    
    # Train Ensemble
    model = get_ensemble_model()
    model.fit(X_train_vec, y_train_enc)
    
    # Predict
    y_pred_enc = model.predict(X_test_vec)
    y_pred = le.inverse_transform(y_pred_enc)
    y_test = test_df['label_full']
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    print(f"   Accuracy: {acc:.4f}")
    
    # Save Classification Report
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred, labels=['Pro-Palestine', 'Pro-Israel', 'Neutral'])
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Pro-P', 'Pro-I', 'Neutral'],
                yticklabels=['Pro-P', 'Pro-I', 'Neutral'])
    plt.title(f'Ensemble Model Confusion Matrix: {train_name} -> {test_name}\nAccuracy: {acc:.2%}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    try:
        plt.show()
        print(f"   âœ“ Saved confusion matrix: cm_{train_name}_to_{test_name}.png")
    except Exception as e:
        print(f"âš ï¸ Could not save plot: {e}")
    finally:
        plt.close()
    
    return model, tfidf, le


In [ ]:
# ============================================================================
# 4. EXPERIMENT 1: WITHIN-PLATFORM SPLIT


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("EXPERIMENT 1: WITHIN-PLATFORM PREDICTION")
print("-"*(60))

# Reddit Split
r_train, r_test = train_test_split(reddit_df, test_size=0.2, random_state=42, stratify=reddit_df['label_full'])
reddit_model, reddit_tfidf, reddit_le = train_evaluate_model(r_train, r_test, "Reddit", "Reddit")

# YouTube Split
y_train, y_test = train_test_split(youtube_df, test_size=0.2, random_state=42, stratify=youtube_df['label_full'])
youtube_model, youtube_tfidf, youtube_le = train_evaluate_model(y_train, y_test, "YouTube", "YouTube")


In [ ]:
# ============================================================================
# 5. EXPERIMENT 2: CROSS-PLATFORM PREDICTION


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("EXPERIMENT 2: CROSS-PLATFORM GENERALIZATION")
print("-"*(60))

# Train Reddit -> Test YouTube
train_evaluate_model(reddit_df, youtube_df, "Reddit", "YouTube")

# Train YouTube -> Test Reddit
train_evaluate_model(youtube_df, reddit_df, "YouTube", "Reddit")


In [ ]:
# ============================================================================
# 6. FEATURE IMPORTANCE ANALYSIS


In [ ]:
# ============================================================================
print("\n" + "-"*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("-"*(60))

def plot_feature_importance(model, tfidf, le, title_prefix):
    # Extract the Logistic Regression estimator from the VotingClassifier
    # It is the first estimator ('lr')
    lr_model = model.estimators_[0]
    
    feature_names = tfidf.get_feature_names_out()
    classes = le.classes_
    
    # Get top 15 features for each class
    top_features = {}
    
    for i, class_label in enumerate(classes):
        # For binary, coef_ is (1, n_features), for multi-class it is (n_classes, n_features)
        if len(classes) == 2:
            coefs = lr_model.coef_[0] if i == 1 else -lr_model.coef_[0]
        else:
            coefs = lr_model.coef_[i]
            
        top_indices = np.argsort(coefs)[-15:] # Top 15
        top_words = [(feature_names[j], coefs[j]) for j in top_indices]
        top_features[class_label] = top_words
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    fig.suptitle(f'{title_prefix}: Top Predictive Keywords (Ensemble Model)', fontsize=16, fontweight='bold')
    
    colors = {'Pro-Palestine': '#2ecc71', 'Pro-Israel': '#3498db', 'Neutral': '#95a5a6'}
    
    for i, class_label in enumerate(['Pro-Palestine', 'Pro-Israel', 'Neutral']):
        if class_label in top_features:
            words, scores = zip(*top_features[class_label])
            axes[i].barh(words, scores, color=colors.get(class_label, 'gray'))
            axes[i].set_title(class_label, fontsize=14)
            axes[i].set_xlabel('Coefficient Magnitude')
    
    plt.tight_layout()
    try:
        plt.show()
        print(f"âœ“ Saved feature importance: features_{title_prefix.lower()}.png")
    except Exception as e:
        print(f"âš ï¸ Could not save feature plot: {e}")
    finally:
        plt.close()

# Analyze features for the Reddit model
plot_feature_importance(reddit_model, reddit_tfidf, reddit_le, "Reddit")

# Analyze features for the YouTube model
plot_feature_importance(youtube_model, youtube_tfidf, youtube_le, "YouTube")

print("\n" + "="*80)
print("âœ… ML ANALYSIS COMPLETE")
print("="*80)


In [ ]:
from scipy.stats import mannwhitneyu, kruskal

# Response time differences: Mann-Whitney U test
if 'response_time_mins' in reddit_df.columns:
    u, p = mannwhitneyu(reddit_df['response_time_mins'].dropna(), youtube_df['response_time_mins'].dropna() if 'response_time_mins' in youtube_df.columns else [1], alternative='two-sided')
    print(f"Response time difference: U={u:.0f}, p={p:.6f}")

# Echo chamber metrics by stance
if 'Label' in reddit_df.columns:
    stances = reddit_df['Label'].unique()
    print("\nEcho Chamber Index by Stance:")
    for stance in sorted(stances):
        subset = reddit_df[reddit_df['Label'] == stance]
        print(f"  Stance {stance}: n={len(subset)}")


## Statistical Significance Tests

Mann-Whitney U tests for platform differences and Kruskal-Wallis tests for stance effects.



In [ ]:
# Export echo chamber and network analysis results
print("\n" + "=" * 80)
print("EXPORTING ECHO CHAMBER ANALYSIS DATA")
print("=" * 80)

if 'author_name' in reddit_df.columns or 'author' in youtube_df.columns:
    # Export network analysis results
    echo_reddit = reddit_df[[col for col in reddit_df.columns 
                             if col in ['author_name', 'Label', 'vader_label', 'response_time_mins']]].copy()
    echo_youtube = youtube_df[[col for col in youtube_df.columns 
                               if col in ['author', 'Label', 'vader_label']]].copy()
    
    reddit_echo_path = OUTPUT_DIR / 'reddit_echo_chambers.csv'
    youtube_echo_path = OUTPUT_DIR / 'youtube_echo_chambers.csv'
    
    echo_reddit.to_csv(reddit_echo_path, index=False, encoding='utf-8')
    echo_youtube.to_csv(youtube_echo_path, index=False, encoding='utf-8')
    
    print(f"\n✅ Exported: {reddit_echo_path}")
    print(f"✅ Exported: {youtube_echo_path}")
    print(f"\n📌 Echo chamber analysis complete. Network visualizations displayed above.")


## Export Echo Chamber Analysis Results

Save network and ML prediction results.

